<a href="https://colab.research.google.com/github/narame7/UOS-FootballDataAnalytics-Tutorial/blob/main/Week%202/1-load-statistic-data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 통계 데이터 크롤링

In [ ]:
!pip install ScraperFC soccerdata

In [ ]:
import pandas as pd
import ScraperFC as sfc
import soccerdata as sd

## 필요한 라이브러리 import

## [ClubElo](http://clubelo.com/)
- 팀들의 역대 Elo 점수를 알 수 있는 사이트.


In [ ]:
ce = sfc.ClubElo()
try:
    print(ce.scrape_team_on_date('Tottenham', '2023-08-13'))
except Exception as e:
    # ClubElo API(api.clubelo.com) 서버가 응답하지 않으면 빈 응답이 와서 오류가 납니다.
    print('ClubElo API에서 데이터를 받지 못했습니다. 잠시 후 다시 시도해 보세요.', repr(e))

## [Understat](https://understat.com/)

- 자체 xG 모델과 여러 기록이 있는 통계 사이트
- Understat 사이트 구조가 바뀌어 ScraperFC(4.5.0)의 Understat 스크레이퍼는 동작하지 않습니다. 대신 [soccerdata](https://soccerdata.readthedocs.io/en/latest/datasources/Understat.html)의 Understat 리더를 사용합니다.

In [ ]:
us = sd.Understat(leagues="ENG-Premier League", seasons="2026/2027")

### 경기 일정과 링크 불러오기

In [ ]:
schedule = us.read_schedule().reset_index()
us_match_links = schedule['url'].tolist()
schedule[['date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'url']].head()

### 팀 순위 보기

In [ ]:
team_match = us.read_team_match_stats().reset_index()

# 경기 단위 기록을 팀 단위(홈/원정)로 풀어서 순위표 만들기
cols = ['team', 'points', 'goals_for', 'goals_against', 'xg', 'xga']
home_rows = team_match[['home_team', 'home_points', 'home_goals', 'away_goals', 'home_xg', 'away_xg']].set_axis(cols, axis=1).assign(venue='Home')
away_rows = team_match[['away_team', 'away_points', 'away_goals', 'home_goals', 'away_xg', 'home_xg']].set_axis(cols, axis=1).assign(venue='Away')
team_rows = pd.concat([home_rows, away_rows], ignore_index=True)

def league_table(df):
    table = df.groupby('team').agg(
        M=('points', 'size'), Pts=('points', 'sum'),
        GF=('goals_for', 'sum'), GA=('goals_against', 'sum'),
        xG=('xg', 'sum'), xGA=('xga', 'sum'),
    )
    table['GD'] = table['GF'] - table['GA']
    return table.sort_values(['Pts', 'GD', 'GF'], ascending=False)

tables = [league_table(team_rows),
          league_table(team_rows[team_rows['venue'] == 'Home']),
          league_table(team_rows[team_rows['venue'] == 'Away'])]

In [ ]:
tables[0] # 전체 순위, 홈 경기 순위, 원정 경기 순위

### 팀 내 순위 보기

In [ ]:
player_stats = us.read_player_season_stats().reset_index()

In [ ]:
player_stats['team'].unique()

In [ ]:
everton = player_stats[player_stats['team'] == 'Everton']
everton[['player', 'position', 'matches', 'minutes', 'goals', 'xg', 'assists', 'xa', 'shots', 'key_passes']].sort_values('xg', ascending=False).head()

### 경기 정보 보기

In [ ]:
first_match = schedule.iloc[0]
us_match = us.read_player_match_stats(match_id=int(first_match['game_id'])).reset_index()

In [ ]:
us_match['team'].unique() # home away 구분

In [ ]:
us_match_df = us_match[us_match['team'] == first_match['home_team']] # 첫번째 경기의 홈 팀 정보

In [ ]:
us_match_df.head()

### 여러 경기의 슈팅 데이터 보기

In [ ]:
match_ids = schedule['game_id'].iloc[:5].astype(int).tolist() # 처음 5경기
us_shots = us.read_shot_events(match_id=match_ids)

In [ ]:
us_shots.reset_index()['game'].value_counts()

In [ ]:
us_shots.head()

### 시즌 누적 데이터 불러오기

In [ ]:
player_stats.sort_values('xg', ascending=False).head(10) # 시즌 xG 상위 10명

### 특정 팀의 경기 데이터 불러오기

In [ ]:
team_name = 'Aston Villa'
us_team_data = team_match[(team_match['home_team'] == team_name) | (team_match['away_team'] == team_name)]
us_team_data[['date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'home_ppda', 'away_ppda']].head() # 특정 팀(aston villa)의 경기 기록

### [SoccerData](https://soccerdata.readthedocs.io/en/latest/)

In [33]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_context("notebook")
sns.set_style("whitegrid")

#### Evolution of top team’s Elo ratings

In [ ]:
# How did the current top 5 teams in the world develop over time?
elo = sd.ClubElo()
try:
    current_elo = elo.read_by_date()
except Exception as e:
    # ClubElo API(api.clubelo.com) 서버가 응답하지 않으면 다음 셀은 건너뜁니다.
    current_elo = None
    print('ClubElo API에서 데이터를 받지 못했습니다. 잠시 후 다시 시도해 보세요.', repr(e))
current_elo.head() if current_elo is not None else None

In [ ]:
if current_elo is not None:
    num_teams = 5
    smoothing = 100
    elo_top_development = pd.concat(
        [
            elo.read_team_history(team)["elo"].rolling(smoothing).mean()
            for team in current_elo.reset_index()["team"][:num_teams]
        ],
        axis=1,
    )
    elo_top_development.columns = current_elo.reset_index()["team"][:num_teams]
    elo_top_development = elo_top_development.ffill()

    fig = plt.figure(figsize=(16, 10))
    ax1 = fig.add_subplot(111, ylabel="ELO rolling avg.", xlabel="Date")
    elo_top_development.plot(ax=ax1)
    ax1.legend(loc="upper left", frameon=False, bbox_to_anchor=(0, 1.05), ncol=num_teams)
    sns.despine();

#### Home team advantage in the Italian Serie A

In [ ]:
# We all know sports teams have an advantage when playing at home. Here’s a look at home team advantage for 5 years of the Serie A.
try:
    seriea_hist = sd.MatchHistory("ITA-Serie A", range(2018, 2023))
    games = seriea_hist.read_games()
except Exception as e:
    # football-data.co.uk 서버가 응답하지 않으면 아래 홈 어드밴티지 셀들은 건너뜁니다.
    games = None
    print('football-data.co.uk에서 데이터를 받지 못했습니다. 잠시 후 다시 시도해 보세요.', repr(e))
games.sample(5) if games is not None else None

In [37]:
def home_away_results(games: pd.DataFrame):
    """Returns aggregated home/away results per team"""
    res = pd.melt(
        games.reset_index(),
        id_vars=["date", "FTR"],
        value_name="team",
        var_name="is_home",
        value_vars=["home_team", "away_team"],
    )

    res.is_home = res.is_home.replace(["home_team", "away_team"], ["Home", "Away"])
    res["win"] = res["lose"] = res["draw"] = 0
    res.loc[(res["is_home"] == "Home") & (res["FTR"] == "H"), "win"] = 1
    res.loc[(res["is_home"] == "Away") & (res["FTR"] == "A"), "win"] = 1
    res.loc[(res["is_home"] == "Home") & (res["FTR"] == "A"), "lose"] = 1
    res.loc[(res["is_home"] == "Away") & (res["FTR"] == "H"), "lose"] = 1
    res.loc[res["FTR"] == "D", "draw"] = 1

    groups = res.groupby(["team", "is_home"])
    win = groups.win.agg(["sum", "mean"]).rename(columns={"sum": "n_win", "mean": "win_pct"})
    loss = groups.lose.agg(["sum", "mean"]).rename(columns={"sum": "n_lose", "mean": "lose_pct"})
    draw = groups.draw.agg(["sum", "mean"]).rename(columns={"sum": "n_draw", "mean": "draw_pct"})

    res = pd.concat([win, loss, draw], axis=1)
    return res

In [ ]:
results = home_away_results(games) if games is not None else None
results.head(6) if results is not None else None

In [ ]:
# The overall picture shows most teams have a clear advantage at home:
if results is not None:
    g = sns.FacetGrid(results.reset_index(), hue="team", palette="Set2", height=6, aspect=0.5)
    g.map(sns.pointplot, "is_home", "win_pct", order=["Away", "Home"])
    g.set_axis_labels("", "win %");

In [ ]:
# But there are a few exceptions
if results is not None:
    g = sns.FacetGrid(results.reset_index(), col="team", col_wrap=5)
    g.map(sns.pointplot, "is_home", "win_pct", order=["Away", "Home"])
    g.set_axis_labels("", "win %");